
# Phase 1 — Dataset Understanding & Problem Definition

**Objective:** Establish exactly what data you have and quantify the imbalance problem.

This notebook performs the activities outlined in Phase 1 of the VAE-GAN-based framework for generating synthetic samples of minority network attack categories in the CICIDS2017 dataset.


In [ ]:
# ratio for different types of attack

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Define data path
DATA_DIR = "./data/CICIDS2017/"
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {os.path.exists(DATA_DIR)}")


Data directory: ./data/CICIDS2017/
Directory exists: True


In [2]:

# List all CSV files in the data directory
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  - {f}")


Found 8 CSV file(s):
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Wednesday-workingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv


In [4]:

# Load and concatenate all CSV files
df_list = []
for file in csv_files:
    file_path = os.path.join(DATA_DIR, file)
    print(f"Loading {file_path}...")
    df = pd.read_csv(file_path, encoding="cp1252")
    df_list.append(df)
    print(f"  Shape: {df.shape}")

if len(df_list) > 0:
    df = pd.concat(df_list, ignore_index=True)
    print(f"Combined dataset shape: {df.shape}")
else:
    # If no CSV files found, create an empty dataframe for demonstration
    print("No CSV files found. Creating an empty dataframe for demonstration.")
    df = pd.DataFrame()

Loading ./data/CICIDS2017/Tuesday-WorkingHours.pcap_ISCX.csv...
  Shape: (445909, 85)
Loading ./data/CICIDS2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
  Shape: (286467, 85)
Loading ./data/CICIDS2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
  Shape: (458968, 85)
Loading ./data/CICIDS2017/Friday-WorkingHours-Morning.pcap_ISCX.csv...
  Shape: (191033, 85)
Loading ./data/CICIDS2017/Wednesday-workingHours.pcap_ISCX.csv...
  Shape: (692703, 85)
Loading ./data/CICIDS2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
  Shape: (288602, 85)
Loading ./data/CICIDS2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
  Shape: (225745, 85)
Loading ./data/CICIDS2017/Monday-WorkingHours.pcap_ISCX.csv...
  Shape: (529918, 85)
Combined dataset shape: (3119345, 85)


In [6]:
df.shape

(3119345, 85)

In [15]:
df.columns = df.columns.str.strip()

In [16]:
df.columns

Index(['Flow ID', 'Source IP', 'Source Port', 'Destination IP',
       'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration',
       'Total Fwd Packets', 'Total Backward Packets',
       'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
  

In [17]:
if 'Label' in df.columns:
    # Identify all attack categories (excluding 'BENIGN')
    attack_categories = df[df['Label'] != 'BENIGN']['Label'].unique()
    benign_count = df[df['Label'] == 'BENIGN'].shape[0]
    print(f"BENIGN samples: {benign_count}")
    print(f"Attack categories found: {list(attack_categories)}")
    print(f"Number of attack categories: {len(attack_categories)}")
    
    # Count samples per category
    class_counts = df['Label'].value_counts()
    print("Class distribution:")
    display(class_counts)
    
    # Calculate percentages
    class_percentages = (df['Label'].value_counts(normalize=True) * 100).round(2)
    print("Class percentages:")
    display(class_percentages)
    
    # Identify majority and minority classes
    majority_class = class_counts.index[0]
    majority_count = class_counts.iloc[0]
    print(f"Majority class: '{majority_class}' with {majority_count} samples")
    
    # Calculate imbalance ratios
    imbalance_ratios = {}
    for label, count in class_counts.items():
        if label != majority_class:
            ratio = majority_count / count
            imbalance_ratios[label] = ratio
    
    print("Imbalance ratios (majority/minority):")
    imbalance_df = pd.DataFrame.from_dict(imbalance_ratios, orient='index', columns=['Imbalance Ratio'])
    imbalance_df = imbalance_df.sort_values('Imbalance Ratio', ascending=False)
    display(imbalance_df)
    
    # Save class distribution report
    class_dist_df = pd.DataFrame({
        'Label': class_counts.index,
        'Count': class_counts.values,
        'Percentage': class_percentages.values
    })
    os.makedirs('./reports', exist_ok=True)
    class_dist_df.to_csv('./reports/class_distribution_report.csv', index=False)
    print("Class distribution report saved to './reports/class_distribution_report.csv'")
else:
    print("Label column not found in the dataset.")


BENIGN samples: 2273097
Attack categories found: ['FTP-Patator', 'SSH-Patator', 'PortScan', 'Web Attack – Brute Force', 'Web Attack – XSS', 'Web Attack – Sql Injection', nan, 'Bot', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed', 'Infiltration', 'DDoS']
Number of attack categories: 15
Class distribution:


Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack – Brute Force         1507
Web Attack – XSS                  652
Infiltration                       36
Web Attack – Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Class percentages:


Label
BENIGN                        80.30
DoS Hulk                       8.16
PortScan                       5.61
DDoS                           4.52
DoS GoldenEye                  0.36
FTP-Patator                    0.28
SSH-Patator                    0.21
DoS slowloris                  0.20
DoS Slowhttptest               0.19
Bot                            0.07
Web Attack – Brute Force       0.05
Web Attack – XSS               0.02
Infiltration                   0.00
Web Attack – Sql Injection     0.00
Heartbleed                     0.00
Name: proportion, dtype: float64

Majority class: 'BENIGN' with 2273097 samples
Imbalance ratios (majority/minority):


,Imbalance Ratio
Heartbleed,206645.181818
Web Attack – Sql Injection,108242.714286
Infiltration,63141.583333
Web Attack – XSS,3486.345092
Web Attack – Brute Force,1508.358991
Bot,1156.203967
DoS Slowhttptest,413.365521
DoS slowloris,392.183747
SSH-Patator,385.466678
FTP-Patator,286.356387


Class distribution report saved to './reports/class_distribution_report.csv'


In [ ]:

if 'Label' in df.columns and df.shape[0] > 0:
    # Define minority attack categories based on relative threshold (< 1% of total samples)
    total_samples = df.shape[0]
    class_percentages = (df['Label'].value_counts(normalize=True) * 100).round(2)
    minority_threshold = 1.0  # less than 1%
    minority_categories = class_percentages[class_percentages < minority_threshold].index.tolist()
    # Also consider absolute count threshold (e.g., < 100 samples) for very small datasets
    absolute_threshold = 100
    minority_categories_abs = df['Label'].value_counts()[df['Label'].value_counts() < absolute_threshold].index.tolist()
    # Combine both criteria (union)
    minority_categories = list(set(minority_categories + minority_categories_abs))
    # Remove 'BENIGN' if present
    if 'BENIGN' in minority_categories:
        minority_categories.remove('BENIGN')
    
    print(f"Total samples: {total_samples}")
    print(f"Minority categories (using <{minority_threshold}% or <{absolute_threshold} absolute count): {minority_categories}")
    print(f"Number of minority categories: {len(minority_categories)}")
    
    # Save minority-class definition
    with open('./reports/minority_class_definition.txt', 'w') as f:
        f.write("Minority-Class Definition\n")
        f.write("="*50 + "\n")
        f.write(f"Total samples: {total_samples}\n")
        f.write(f"Criteria: < {minority_threshold}% of total samples OR < {absolute_threshold} absolute count\n")
        f.write(f"Minority attack categories: {minority_categories}\n")
        f.write(f"Number of minority categories: {len(minority_categories)}\n")
    print("Minority-class definition saved to './reports/minority_class_definition.txt'")
else:
    print("Cannot define minority classes due to missing data or Label column.")


In [ ]:

if df.shape[0] > 0 and 'Label' in df.columns:
    # Exclude the Label column for feature analysis
    feature_cols = [col for col in df.columns if col != 'Label']
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    print(f"Number of numeric features: {len(numeric_cols)}")
    print(f"Numeric features: {numeric_cols}")
    
    if len(numeric_cols) > 0:
        # Compute basic statistics for numeric features
        stats_df = df[numeric_cols].describe().transpose()
        stats_df['range'] = stats_df['max'] - stats_df['min']
        stats_df['zero_variance'] = (stats_df['std'] == 0)
        print("\nStatistics for numeric features:")
        display(stats_df[['min', 'max', 'range', 'mean', 'std', 'zero_variance']])
        
        # Identify features with zero variance
        zero_var_cols = stats_df[stats_df['zero_variance']].index.tolist()
        if zero_var_cols:
            print(f"\nFeatures with zero variance: {zero_var_cols}")
        else:
            print("\nNo features with zero variance.")
        
        # Check for negative values where they might not be expected
        # For example, packet counts, durations should be non-negative
        # We'll check a few known feature names if they exist
        non_negative_features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                                 'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
                                 'Fwd Packet Length Max', 'Fwd Packet Length Min',
                                 'Bwd Packet Length Max', 'Bwd Packet Length Min']
        unexpected_negative = []
        for feat in non_negative_features:
            if feat in df.columns:
                neg_count = (df[feat] < 0).sum()
                if neg_count > 0:
                    unexpected_negative.append((feat, neg_count))
        if unexpected_negative:
            print("\nUnexpected negative values found:")
            for feat, count in unexpected_negative:
                print(f"  {feat}: {count} negative values")
        else:
            print("\nNo unexpected negative values in common non-negative features.")
    else:
        print("No numeric features found.")
else:
    print("Cannot analyze features due to missing data or Label column.")


In [ ]:

if df.shape[0] > 0:
    print("Data Quality Report")
    print("="*50)
    
    # Missing values
    missing_counts = df.isnull().sum()
    missing_percentages = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing_counts,
        'Missing Percentage': missing_percentages.round(2)
    })
    print("\nMissing values:")
    display(missing_df[missing_df['Missing Count'] > 0])
    
    # Infinite values in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    infinite_counts = pd.Series(0, index=numeric_cols)
    for col in numeric_cols:
        infinite_counts[col] = np.isinf(df[col]).sum()
    infinite_df = pd.DataFrame({
        'Infinite Count': infinite_counts
    })
    print("\nInfinite values:")
    display(infinite_df[infinite_df['Infinite Count'] > 0])
    
    # Duplicated rows
    duplicated_count = df.duplicated().sum()
    print(f"\nDuplicated rows: {duplicated_count}")
    
    # Anomalous values (based on domain knowledge for common features)
    # We'll check a few features if they exist
    anomalous_findings = []
    # Flow Duration should be non-negative
    if 'Flow Duration' in df.columns:
        neg_duration = (df['Flow Duration'] < 0).sum()
        if neg_duration > 0:
            anomalous_findings.append(('Flow Duration', 'negative values', neg_duration))
    # Total Fwd Packets should be non-negative integer
    if 'Total Fwd Packets' in df.columns:
        neg_fwd = (df['Total Fwd Packets'] < 0).sum()
        if neg_fwd > 0:
            anomalous_findings.append(('Total Fwd Packets', 'negative values', neg_fwd))
        non_int_fwd = (~df['Total Fwd Packets'].apply(lambda x: isinstance(x, (int, np.integer)))).sum()
        # Actually, the column might be float, so we check if it's close to integer
        # We'll skip this for simplicity
    # Save data quality report
    with open('./reports/initial_data_quality_report.txt', 'w') as f:
        f.write("Initial Data Quality Report\n")
        f.write("="*50 + "\n\n")
        f.write("Missing Values:\n")
        f.write(missing_df.to_string() + "\n\n")
        f.write("Infinite Values:\n")
        f.write(infinite_df.to_string() + "\n\n")
        f.write(f"Duplicated Rows: {duplicated_count}\n\n")
        f.write("Anomalous Values:\n")
        if anomalous_findings:
            for feat, anomaly, count in anomalous_findings:
                f.write(f"  {feat}: {anomaly} = {count}\n")
        else:
            f.write("  No anomalous values found in checked features.\n")
    print("\nInitial data quality report saved to './reports/initial_data_quality_report.txt'")
else:
    print("No data to perform quality check.")


In [ ]:

if 'Label' in df.columns and df.shape[0] > 0:
    plt.figure(figsize=(12, 6))
    class_counts = df['Label'].value_counts()
    # Use log scale if there is a large imbalance
    ax = sns.barplot(x=class_counts.index, y=class_counts.values)
    plt.title('Class Distribution (Log Scale)')
    plt.yscale('log')
    plt.xlabel('Attack Category')
    plt.ylabel('Number of Samples (Log Scale)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    os.makedirs('./reports/figures', exist_ok=True)
    plt.savefig('./reports/figures/class_distribution_log.png', dpi=300)
    plt.show()
    
    # Also linear scale for minority classes only (excluding majority)
    majority_class = class_counts.index[0]
    minority_counts = class_counts[class_counts.index != majority_class]
    if len(minority_counts) > 0:
        plt.figure(figsize=(12, 6))
        ax = sns.barplot(x=minority_counts.index, y=minority_counts.values)
        plt.title('Class Distribution of Minority Attack Categories')
        plt.xlabel('Attack Category')
        plt.ylabel('Number of Samples')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('./reports/figures/minority_class_distribution.png', dpi=300)
        plt.show()
    print("Class distribution plots saved to ./reports/figures/")
else:
    print("No data to visualize.")



## Conclusion

In this phase, we have:

1. Loaded the CICIDS2017/CICFlowMeter flow data (or a synthetic placeholder).
2. Identified all attack categories and the BENIGN class.
3. Analyzed the number of samples per category and calculated class imbalance ratios.
4. Defined minority attack categories based on relative and absolute thresholds.
5. Analyzed feature types, ranges, and checked for missing, infinite, duplicated, and anomalous values.
6. Generated reports and visualizations summarizing the dataset profile, class distribution, minority-class definition, and initial data quality.

## Next Steps

Proceed to Phase 2: Design and implement the VAE-GAN framework for generating synthetic samples of the minority attack categories identified in this phase.
